In [1]:
import os
import sys
import yaml
import numpy as np
import pandas as pd
import duckdb
import rasterio
import geopandas as gpd
import shapely
import h3
import xarray as xr

# 1. Define project root and paths
PROJECT_ROOT = r"C:\Users\aseem\OneDrive\Dokumen\aerospatium"
RAW_DATA_DIR = os.path.join(PROJECT_ROOT, "data", "raw")
PROCESSED_DATA_DIR = os.path.join(PROJECT_ROOT, "data", "processed")
CONFIGS_DIR = os.path.join(PROJECT_ROOT, "configs")

os.makedirs(PROCESSED_DATA_DIR, exist_ok=True)
os.makedirs(CONFIGS_DIR, exist_ok=True)

# 2. Write/Verify canonical configs/base.yaml
base_config = {
    "project_name": "AeroSpatium",
    "study_area": {
        "name": "Butte_County_Camp_Fire_AOI",
        "bbox": {
            "min_lon": -121.8,
            "min_lat": 39.6,
            "max_lon": -121.3,
            "max_lat": 40.0
        },
        "crs": "EPSG:4326",
        "h3_resolution": 8
    },
    "scenario": {
        "canonical_date": "2023-08-15",
        "simulation_start_utc": "2023-08-15T00:00:00Z",
        "simulation_end_utc": "2023-08-15T23:00:00Z"
    },
    "paths": {
        "raw_dir": "data/raw",
        "processed_dir": "data/processed",
        "h3_static_output": "data/processed/h3_static_features.parquet"
    }
}

config_path = os.path.join(CONFIGS_DIR, "base.yaml")
with open(config_path, "w") as f:
    yaml.dump(base_config, f, default_flow_style=False)

print(f"Base configuration written to: {config_path}")

# 3. File existence sanity check in data/raw
expected_files = {
    "DEM": os.path.join(RAW_DATA_DIR, "dem", "srtm_30m_aoi.tif"),
    "LANDFIRE_EVT": os.path.join(RAW_DATA_DIR, "landfire", "landfire_evt.tif"),
    "FUEL_CROSSWALK": os.path.join(RAW_DATA_DIR, "landfire", "fuel_crosswalk.csv"),
    "USDM": os.path.join(RAW_DATA_DIR, "drought", "USDM_20260818.shp"),
    "AIRNOW": os.path.join(RAW_DATA_DIR, "airnow", "airnow_pm25_aoi.csv"),
    "CENSUS": os.path.join(RAW_DATA_DIR, "census", "census_blocks.geojson"),
    "WEATHER": os.path.join(RAW_DATA_DIR, "weather", "era5_hourly_aoi.nc"),
    "MODIS": os.path.join(RAW_DATA_DIR, "modis", "modis_2023_usa.csv"),
    "WFIGS": os.path.join(RAW_DATA_DIR, "wfigs", "wfigs_perimeters.geojson")
}

print("\n--- Physical Data Source Check ---")
for key, path in expected_files.items():
    exists = os.path.exists(path)
    size_mb = os.path.getsize(path) / (1024 * 1024) if exists else 0.0
    status = "EXISTS" if exists else "MISSING"
    print(f"[{status}] {key:<16}: {size_mb:8.2f} MB -> {os.path.basename(path)}")

print("\nStep 1 complete. Ready for Step 2.")

Base configuration written to: C:\Users\aseem\OneDrive\Dokumen\aerospatium\configs\base.yaml

--- Physical Data Source Check ---
[EXISTS] DEM             :     9.90 MB -> srtm_30m_aoi.tif
[EXISTS] LANDFIRE_EVT    :     4.95 MB -> landfire_evt.tif
[EXISTS] FUEL_CROSSWALK  :     0.00 MB -> fuel_crosswalk.csv
[EXISTS] USDM            :     4.02 MB -> USDM_20260818.shp
[EXISTS] AIRNOW          :     0.00 MB -> airnow_pm25_aoi.csv
[EXISTS] CENSUS          :     0.00 MB -> census_blocks.geojson
[EXISTS] WEATHER         :     0.05 MB -> era5_hourly_aoi.nc
[EXISTS] MODIS           :     7.22 MB -> modis_2023_usa.csv
[EXISTS] WFIGS           :     0.00 MB -> wfigs_perimeters.geojson

Step 1 complete. Ready for Step 2.


In [2]:
import h3
import pandas as pd
import geopandas as gpd
from shapely.geometry import Polygon, Point, box

# 1. Create AOI bounding box polygon
min_lon, min_lat, max_lon, max_lat = -121.8, 39.6, -121.3, 40.0
aoi_box = box(min_lon, min_lat, max_lon, max_lat)

# 2. Polygon to H3 cells using h3-py (v4 API)
# Convert shapely polygon exterior coordinates to lat/lon tuples for h3
aoi_geo_boundary = list(aoi_box.exterior.coords)
# Note: h3 v4 expects LatLngPoly with [(lat, lon), ...]
h3_polygon = h3.LatLngPoly([(lat, lon) for lon, lat in aoi_geo_boundary])
h3_cells = sorted(list(h3.polygon_to_cells(h3_polygon, 8)))

print(f"Total H3 Resolution 8 cells generated for AOI: {len(h3_cells)}")

# 3. Build base H3 GeoDataFrame with centroids and geometry WKT
h3_records = []
for h_str in h3_cells:
    h_u64 = h3.str_to_int(h_str)
    lat, lon = h3.cell_to_latlng(h_str)
    
    # Get boundary polygon vertices (lat, lon) and convert to (lon, lat)
    boundary = h3.cell_to_boundary(h_str)
    poly = Polygon([(b_lon, b_lat) for b_lat, b_lon in boundary])
    
    h3_records.append({
        "h3_index_str": h_str,
        "h3_id": np.uint64(h_u64),
        "centroid_lon": np.float32(lon),
        "centroid_lat": np.float32(lat),
        "geometry": poly,
        "geometry_wkt": poly.wkt
    })

gdf_h3 = gpd.GeoDataFrame(h3_records, geometry="geometry", crs="EPSG:4326")

print("\n--- H3 Grid Verification ---")
print(f"DataFrame shape: {gdf_h3.shape}")
print(f"h3_id dtype: {gdf_h3['h3_id'].dtype}")
print(f"Sample h3_id uint64: {gdf_h3['h3_id'].iloc[0]}")
print(f"Centroid Lon range: [{gdf_h3['centroid_lon'].min():.4f}, {gdf_h3['centroid_lon'].max():.4f}]")
print(f"Centroid Lat range: [{gdf_h3['centroid_lat'].min():.4f}, {gdf_h3['centroid_lat'].max():.4f}]")

# Quick check on sliver cells against raster eastern/southern bounds
RASTER_MAX_LON = -121.3014
RASTER_MIN_LAT = 39.60112
sliver_mask = (gdf_h3["centroid_lon"] > RASTER_MAX_LON) | (gdf_h3["centroid_lat"] < RASTER_MIN_LAT)
print(f"Identified boundary sliver cells: {sliver_mask.sum()} / {len(gdf_h3)}")

Total H3 Resolution 8 cells generated for AOI: 2526

--- H3 Grid Verification ---
DataFrame shape: (2526, 6)
h3_id dtype: uint64
Sample h3_id uint64: 613194612432961535
Centroid Lon range: [-121.7998, -121.3000]
Centroid Lat range: [39.6002, 39.9999]
Identified boundary sliver cells: 15 / 2526


In [3]:
import rasterio
from rasterio.transform import rowcol
import numpy as np
import pandas as pd
from tqdm.auto import tqdm

# 1. Load DEM and EVT rasters
dem_path = os.path.join(RAW_DATA_DIR, "dem", "srtm_30m_aoi.tif")
evt_path = os.path.join(RAW_DATA_DIR, "landfire", "landfire_evt.tif")
lut_path = os.path.join(RAW_DATA_DIR, "landfire", "fuel_crosswalk.csv")

# Read fuel crosswalk table
df_lut = pd.read_csv(lut_path)
print("--- Fuel Crosswalk LUT ---")
print(df_lut[["evt_code", "fuel_name", "fbfm13", "fuel_load_kg_m2", "fuel_bed_depth_m"]])

with rasterio.open(dem_path) as dem_src, rasterio.open(evt_path) as evt_src:
    assert dem_src.transform == evt_src.transform, "Raster transform mismatch!"
    assert dem_src.shape == evt_src.shape, "Raster shape mismatch!"
    
    dem = dem_src.read(1)  # float32
    evt = evt_src.read(1)  # int16
    transform = dem_src.transform
    crs = dem_src.crs
    height, width = dem.shape

print(f"\nLoaded DEM & EVT rasters: {width} x {height} pixels.")

# 2. Compute Horn / Sobel meter-corrected slope and aspect across DEM
# Approximate cell spacing in meters at latitude ~39.8 deg N
lat_center = 39.8
res_deg = abs(transform[0])
dx_m = res_deg * 111320.0 * np.cos(np.radians(lat_center))  # ~23.6 m
dy_m = res_deg * 110574.0                                   # ~30.6 m

# 2D Gradients: axis=1 is X (columns), axis=0 is Y (rows - note rows decrease in latitude)
dz_dx = np.gradient(dem, dx_m, axis=1)
dz_dy = np.gradient(dem, -dy_m, axis=0)  # negative because row 0 is top (max lat)

slope_rad = np.arctan(np.hypot(dz_dx, dz_dy))
slope_deg = np.degrees(slope_rad).astype(np.float32)

# Aspect: compass direction in radians [0, 2pi), 0 = North, pi/2 = East
aspect_rad = np.mod(np.arctan2(dz_dx, dz_dy), 2 * np.pi).astype(np.float32)

# 3. Aggregate DEM and EVT to H3 cells
# Precompute raster coordinate arrays for pixel-in-cell sampling
# Sampling window around each H3 centroid (~500m radius = ~18 pixels window)
h3_terrain_fuel = []

# Fuel codes to Anderson model mappings
# 1: Short Grass (FM1), 2: Timber Grass (FM2), 4: Chaparral (FM4), 10: Timber Litter (FM10), 11: Non-Burnable (FM99)
GRASS_CODES = {1, 2}
SHRUB_CODES = {4}
TIMBER_CODES = {10}
NONBURN_CODES = {11}

for idx, row in gdf_h3.iterrows():
    c_lon, c_lat = row["centroid_lon"], row["centroid_lat"]
    
    # Pixel row/col for centroid
    r, c = rowcol(transform, c_lon, c_lat)
    
    # Boundary clamp for sliver cells
    r_clamped = np.clip(r, 0, height - 1)
    c_clamped = np.clip(c, 0, width - 1)
    is_raster_backed = (r == r_clamped) and (c == c_clamped)
    
    # 15x15 pixel window around centroid (~450m x 450m approximating hex footprint)
    r_min, r_max = max(0, r_clamped - 7), min(height, r_clamped + 8)
    c_min, c_max = max(0, c_clamped - 7), min(width, c_clamped + 8)
    
    win_dem = dem[r_min:r_max, c_min:c_max]
    win_slope = slope_deg[r_min:r_max, c_min:c_max]
    win_aspect = aspect_rad[r_min:r_max, c_min:c_max]
    win_evt = evt[r_min:r_max, c_min:c_max]
    
    # DEM derivatives
    elev_mean = np.float32(np.mean(win_dem))
    elev_sd = np.float32(np.std(win_dem))
    slope_mean = np.float32(np.mean(win_slope))
    slope_sd = np.float32(np.std(win_slope))
    
    # Circular mean of aspect: atan2(sum(sin), sum(cos))
    sin_sum = np.sum(np.sin(win_aspect))
    cos_sum = np.sum(np.cos(win_aspect))
    aspect_mean = np.float32(np.mod(np.arctan2(sin_sum, cos_sum), 2 * np.pi))
    
    # EVT / Fuel modeling
    unique_vals, counts = np.unique(win_evt, return_counts=True)
    total_px = len(win_evt.flatten())
    
    # Dominant fuel model code
    dominant_evt = unique_vals[np.argmax(counts)]
    fbfm_row = df_lut[df_lut["evt_code"] == dominant_evt]
    fbfm_code = np.int8(fbfm_row["fbfm13"].values[0]) if len(fbfm_row) > 0 else np.int8(99)
    
    # Fractional covers
    frac_grass = np.float32(np.sum([counts[i] for i, v in enumerate(unique_vals) if v in GRASS_CODES]) / total_px)
    frac_shrub = np.float32(np.sum([counts[i] for i, v in enumerate(unique_vals) if v in SHRUB_CODES]) / total_px)
    frac_timber = np.float32(np.sum([counts[i] for i, v in enumerate(unique_vals) if v in TIMBER_CODES]) / total_px)
    
    h3_terrain_fuel.append({
        "h3_id": row["h3_id"],
        "elevation_mean_m": elev_mean,
        "elevation_sd_m": elev_sd,
        "slope_mean_deg": slope_mean,
        "slope_sd_deg": slope_sd,
        "aspect_mean_rad": aspect_mean,
        "fuel_model": fbfm_code,
        "fuel_frac_grass": frac_grass,
        "fuel_frac_shrub": frac_shrub,
        "fuel_frac_timber": frac_timber,
        "raster_backed": is_raster_backed
    })

df_terrain_fuel = pd.DataFrame(h3_terrain_fuel)

# Merge back into main GeoDataFrame
gdf_h3 = gdf_h3.merge(df_terrain_fuel, on="h3_id")

print("\n--- Terrain & Fuel Ingestion Summary ---")
print(f"Total cells: {len(gdf_h3)}")
print(f"Elevation range (m): [{gdf_h3['elevation_mean_m'].min():.1f}, {gdf_h3['elevation_mean_m'].max():.1f}]")
print(f"Slope range (deg): [{gdf_h3['slope_mean_deg'].min():.1f}, {gdf_h3['slope_mean_deg'].max():.1f}]")
print(f"Dominant fuel models distribution:\n{gdf_h3['fuel_model'].value_counts().to_dict()}")
print(f"Raster-backed cells: {gdf_h3['raster_backed'].sum()} / {len(gdf_h3)}")

--- Fuel Crosswalk LUT ---
   evt_code      fuel_name  fbfm13  fuel_load_kg_m2  fuel_bed_depth_m
0         1    Short Grass       1            0.166               0.3
1         2   Timber Grass       2            0.896               0.6
2         4      Chaparral       4            1.120               1.8
3        10  Timber Litter      10            2.690               0.3
4        11   Non-Burnable      99            0.000               0.0

Loaded DEM & EVT rasters: 1800 x 1440 pixels.

--- Terrain & Fuel Ingestion Summary ---
Total cells: 2526
Elevation range (m): [203.8, 1385.2]
Slope range (deg): [4.8, 6.9]
Dominant fuel models distribution:
{1: 848, 2: 819, 4: 811, 10: 48}
Raster-backed cells: 2511 / 2526


In [4]:
import duckdb
import geopandas as gpd
import numpy as np
import pandas as pd
from shapely.geometry import Point

# 1. Ingest USDM Drought via DuckDB Spatial with ST_MakeValid repair
usdm_path = os.path.join(RAW_DATA_DIR, "drought", "USDM_20260818.shp").replace("\\", "/")

con = duckdb.connect()
con.execute("INSTALL spatial;")
con.execute("LOAD spatial;")

# Extract validated polygons
df_usdm = con.execute(f"""
    SELECT DM::TINYINT AS drought_category, 
           ST_AsText(ST_MakeValid(geom)) AS geom_wkt
    FROM ST_Read('{usdm_path}')
""").df()

gdf_usdm = gpd.GeoDataFrame(
    df_usdm, 
    geometry=gpd.GeoSeries.from_wkt(df_usdm["geom_wkt"]), 
    crs="EPSG:4326"
)

# Spatial join H3 centroids to USDM
gdf_centroids = gpd.GeoDataFrame(
    gdf_h3[["h3_id"]],
    geometry=[Point(lon, lat) for lon, lat in zip(gdf_h3["centroid_lon"], gdf_h3["centroid_lat"])],
    crs="EPSG:4326"
)

joined_usdm = gpd.sjoin(gdf_centroids, gdf_usdm[["drought_category", "geometry"]], how="left", predicate="intersects")
# Default to 0 (D0) if point falls outside polygon boundaries
joined_usdm["drought_category"] = joined_usdm["drought_category"].fillna(0).astype(np.int8)
# Deduplicate if a centroid touches polygon boundaries
usdm_map = joined_usdm.groupby("h3_id")["drought_category"].max().to_dict()
gdf_h3["drought_category"] = gdf_h3["h3_id"].map(usdm_map).astype(np.int8)

print("--- USDM Ingestion Complete ---")
print(f"Drought categories distribution: {gdf_h3['drought_category'].value_counts().to_dict()}")

# 2. EPA AirNow PM2.5 baseline IDW interpolation
airnow_path = os.path.join(RAW_DATA_DIR, "airnow", "airnow_pm25_aoi.csv")
df_airnow = pd.read_csv(airnow_path)
print(f"\n--- AirNow Stations ({len(df_airnow)} sites) ---")
print(df_airnow[["site_name", "latitude", "longitude", "pm25_ugm3"]])

# Inverse Distance Weighting (IDW, p=2)
st_lats = df_airnow["latitude"].values
st_lons = df_airnow["longitude"].values
st_pm25 = df_airnow["pm25_ugm3"].values

pm25_interpolated = []
for _, row in gdf_h3.iterrows():
    c_lat, c_lon = row["centroid_lat"], row["centroid_lon"]
    # Great-circle distance approx (flat-Earth projection for local distances in km)
    d_lat = (st_lats - c_lat) * 111.0
    d_lon = (st_lons - c_lon) * 111.0 * np.cos(np.radians(c_lat))
    dist_km = np.sqrt(d_lat**2 + d_lon**2)
    
    # Exact match or very close (< 100m)
    if np.any(dist_km < 0.1):
        pm25_interpolated.append(np.float32(st_pm25[np.argmin(dist_km)]))
    else:
        weights = 1.0 / (dist_km ** 2)
        pm_val = np.sum(weights * st_pm25) / np.sum(weights)
        pm25_interpolated.append(np.float32(pm_val))

gdf_h3["pm25_baseline_ugm3"] = np.array(pm25_interpolated, dtype=np.float32)
print(f"PM2.5 baseline range: [{gdf_h3['pm25_baseline_ugm3'].min():.2f}, {gdf_h3['pm25_baseline_ugm3'].max():.2f}] ug/m3")

# 3. Census Block Group Population Areal Weighting
census_path = os.path.join(RAW_DATA_DIR, "census", "census_blocks.geojson")
gdf_census = gpd.read_file(census_path)
# Project to planar UTM Zone 10N (EPSG:32610) for accurate area computation
gdf_census_utm = gdf_census.to_crs("EPSG:32610")
gdf_h3_utm = gdf_h3[["h3_id", "geometry"]].to_crs("EPSG:32610")

# Compute intersection geometries and areas
intersections = gpd.overlay(gdf_h3_utm, gdf_census_utm, how="intersection")
intersections["intersect_area_m2"] = intersections.geometry.area

# Merge block total area
gdf_census_utm["bg_total_area_m2"] = gdf_census_utm.geometry.area
intersections = intersections.merge(
    gdf_census_utm[["GEOID", "bg_total_area_m2"]], 
    on="GEOID", 
    how="left"
)

# Area-weighted population per intersection piece
intersections["pop_piece"] = intersections["population"] * (intersections["intersect_area_m2"] / intersections["bg_total_area_m2"])

# Aggregate population per H3 cell
h3_pop = intersections.groupby("h3_id")["pop_piece"].sum().to_dict()

# Map back to gdf_h3
gdf_h3["population_total"] = gdf_h3["h3_id"].map(h3_pop).fillna(0.0).round().astype(np.int32)

# Compute population density per km2 (mean H3 res-8 cell area ~0.7523 km2)
HEX_AREA_KM2 = 0.7523
gdf_h3["population_density_km2"] = (gdf_h3["population_total"] / HEX_AREA_KM2).astype(np.float32)

print(f"\n--- Census Population Summary ---")
print(f"Total AOI population allocated: {gdf_h3['population_total'].sum():,} (Census benchmark = {gdf_census['population'].sum():,})")
print(f"Populated H3 cells: {(gdf_h3['population_total'] > 0).sum()} / {len(gdf_h3)}")
print(f"Max population density: {gdf_h3['population_density_km2'].max():.1f} people/km2")

--- USDM Ingestion Complete ---
Drought categories distribution: {0: 2526}

--- AirNow Stations (6 sites) ---
          site_name  latitude  longitude  pm25_ugm3
0  Paradise-Theater    39.759   -121.611       85.4
1   Chico-Manzanita    39.761   -121.843       38.2
2   Oroville-County    39.512   -121.556       22.1
3    Magalia-Skyway    39.812   -121.578      110.6
4    Cohasset-Ridge    39.924   -121.731       45.0
5      Forest-Ranch    39.881   -121.672       64.8
PM2.5 baseline range: [44.26, 110.48] ug/m3

--- Census Population Summary ---
Total AOI population allocated: 8,785 (Census benchmark = 9,200)
Populated H3 cells: 2526 / 2526
Max population density: 6.6 people/km2


In [5]:
import xarray as xr
import pyarrow as pa
import pyarrow.parquet as pq

# 1. Ingest ERA5 Weather baseline at scenario start (2023-08-15 00:00:00 UTC)
weather_path = os.path.join(RAW_DATA_DIR, "weather", "era5_hourly_aoi.nc")
ds_weather = xr.open_dataset(weather_path)

# Handle ascending latitude
ds_weather = ds_weather.sortby("latitude")

# Extract snapshot for baseline surface conditions
t0_weather = ds_weather.sel(time="2023-08-15T00:00:00")

# Compute wind speed (m/s) and relative humidity (%) across the ERA5 grid
u10 = t0_weather["u10"]
v10 = t0_weather["v10"]
t2m = t0_weather["t2m"]
d2m = t0_weather["d2m"]

ws10 = np.sqrt(u10**2 + v10**2)
Tc = t2m - 273.15
Td = d2m - 273.15
rh = 100.0 * np.exp(17.625 * Td / (243.04 + Td)) / np.exp(17.625 * Tc / (243.04 + Tc))

# Bilinear interpolation of initial surface conditions to H3 centroids
points_lat = xr.DataArray(gdf_h3["centroid_lat"].values, dims="points")
points_lon = xr.DataArray(gdf_h3["centroid_lon"].values, dims="points")

gdf_h3["temp_c_t0"] = t2m.interp(latitude=points_lat, longitude=points_lon, method="linear").values.astype(np.float32) - 273.15
gdf_h3["wind_speed_ms_t0"] = ws10.interp(latitude=points_lat, longitude=points_lon, method="linear").values.astype(np.float32)
gdf_h3["rh_pct_t0"] = rh.interp(latitude=points_lat, longitude=points_lon, method="linear").values.astype(np.float32)

print("--- ERA5 Weather Integration Complete ---")
print(f"Temperature at T0 range: [{gdf_h3['temp_c_t0'].min():.1f}, {gdf_h3['temp_c_t0'].max():.1f}] °C")
print(f"Wind speed at T0 range: [{gdf_h3['wind_speed_ms_t0'].min():.2f}, {gdf_h3['wind_speed_ms_t0'].max():.2f}] m/s")
print(f"Relative humidity at T0 range: [{gdf_h3['rh_pct_t0'].min():.1f}, {gdf_h3['rh_pct_t0'].max():.1f}] %")

# 2. Schema Selection and Enforcement
export_columns = [
    "h3_id",
    "centroid_lon",
    "centroid_lat",
    "elevation_mean_m",
    "elevation_sd_m",
    "slope_mean_deg",
    "slope_sd_deg",
    "aspect_mean_rad",
    "fuel_model",
    "fuel_frac_grass",
    "fuel_frac_shrub",
    "fuel_frac_timber",
    "drought_category",
    "pm25_baseline_ugm3",
    "population_total",
    "population_density_km2",
    "temp_c_t0",
    "wind_speed_ms_t0",
    "rh_pct_t0",
    "raster_backed",
    "geometry_wkt"
]

df_export = pd.DataFrame(gdf_h3[export_columns])

# Explicit casting
df_export["h3_id"] = df_export["h3_id"].astype(np.uint64)
df_export["centroid_lon"] = df_export["centroid_lon"].astype(np.float32)
df_export["centroid_lat"] = df_export["centroid_lat"].astype(np.float32)
df_export["elevation_mean_m"] = df_export["elevation_mean_m"].astype(np.float32)
df_export["elevation_sd_m"] = df_export["elevation_sd_m"].astype(np.float32)
df_export["slope_mean_deg"] = df_export["slope_mean_deg"].astype(np.float32)
df_export["slope_sd_deg"] = df_export["slope_sd_deg"].astype(np.float32)
df_export["aspect_mean_rad"] = df_export["aspect_mean_rad"].astype(np.float32)
df_export["fuel_model"] = df_export["fuel_model"].astype(np.int8)
df_export["fuel_frac_grass"] = df_export["fuel_frac_grass"].astype(np.float32)
df_export["fuel_frac_shrub"] = df_export["fuel_frac_shrub"].astype(np.float32)
df_export["fuel_frac_timber"] = df_export["fuel_frac_timber"].astype(np.float32)
df_export["drought_category"] = df_export["drought_category"].astype(np.int8)
df_export["pm25_baseline_ugm3"] = df_export["pm25_baseline_ugm3"].astype(np.float32)
df_export["population_total"] = df_export["population_total"].astype(np.int32)
df_export["population_density_km2"] = df_export["population_density_km2"].astype(np.float32)
df_export["temp_c_t0"] = df_export["temp_c_t0"].astype(np.float32)
df_export["wind_speed_ms_t0"] = df_export["wind_speed_ms_t0"].astype(np.float32)
df_export["rh_pct_t0"] = df_export["rh_pct_t0"].astype(np.float32)
df_export["raster_backed"] = df_export["raster_backed"].astype(bool)
df_export["geometry_wkt"] = df_export["geometry_wkt"].astype(str)

# 3. Validation and Integrity Assertions
assert len(df_export) == 2526, f"Expected 2526 rows, got {len(df_export)}"
assert df_export["h3_id"].nunique() == 2526, "Duplicate H3 IDs detected!"
assert df_export.isnull().sum().sum() == 0, f"Null values found:\n{df_export.isnull().sum()}"
assert (df_export["slope_mean_deg"] >= 0).all(), "Negative slope detected!"
assert (df_export["aspect_mean_rad"] >= 0).all() and (df_export["aspect_mean_rad"] <= 2 * np.pi + 1e-4).all(), "Aspect out of [0, 2pi] bounds!"
assert (df_export["population_total"] >= 0).all(), "Negative population values detected!"

# 4. Parquet Export via PyArrow
output_parquet_path = os.path.join(PROCESSED_DATA_DIR, "h3_static_features.parquet")
table = pa.Table.from_pandas(df_export)
pq.write_table(table, output_parquet_path, compression="snappy")

file_size_kb = os.path.getsize(output_parquet_path) / 1024.0

print("\n=======================================================")
print(f"SUCCESS: Exported H3 Static Features Parquet Table")
print(f"Path: {output_parquet_path}")
print(f"Rows: {len(df_export)}, Columns: {df_export.shape[1]}")
print(f"File Size: {file_size_kb:.2f} KB")
print("=======================================================")
print("\nSample records:")
print(df_export[["h3_id", "elevation_mean_m", "slope_mean_deg", "fuel_model", "population_total", "pm25_baseline_ugm3"]].head(5))

--- ERA5 Weather Integration Complete ---
Temperature at T0 range: [22.0, 36.5] °C
Wind speed at T0 range: [1.54, 7.98] m/s
Relative humidity at T0 range: [17.0, 62.0] %

SUCCESS: Exported H3 Static Features Parquet Table
Path: C:\Users\aseem\OneDrive\Dokumen\aerospatium\data\processed\h3_static_features.parquet
Rows: 2526, Columns: 21
File Size: 412.13 KB

Sample records:
                h3_id  elevation_mean_m  slope_mean_deg  fuel_model  \
0  613194612432961535       1041.642334        6.676373           2   
1  613194612441350143       1046.552612        5.538792           2   
2  613194612456030207       1012.621399        5.724729          10   
3  613194612458127359        995.331665        5.542790           1   
4  613194612460224511       1028.539917        5.154737           1   

   population_total  pm25_baseline_ugm3  
0                 2           71.216248  
1                 4           71.495712  
2                 4           71.581581  
3                 4          